#Classic ML Modeling Guide: Logistic Regression, SVC, and Random Forest

##Setup

###Imports

In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import product

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

colab

In [ ]:
project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
project_path = project_path.resolve()

print("Project path:", project_path)


project paths

In [ ]:
processed_dir = project_path / "datasets" / "processed"
metrics_dir = project_path / "metrics"

train_cleaned_path = processed_dir / "train_cleaned.csv"

classic_metrics_path = metrics_dir / "classic_ml_experiments.csv"
summary_plot_path = metrics_dir / "classic_ml_summary.png"
validation_diagnostics_path = metrics_dir / "classic_ml_validation_diagnostics.csv"

metrics_dir.mkdir(parents=True, exist_ok=True)

###Load Data

In [ ]:
if not train_cleaned_path.exists():
    raise FileNotFoundError(
        f"Missing cleaned training file: {train_cleaned_path}. "
        "Run 01_data_initialization.ipynb first."
    )

data = pd.read_csv(train_cleaned_path)

required_columns = {"label", "headline", "processed_text"}
missing_columns = required_columns - set(data.columns)

if missing_columns:
    raise ValueError(f"Missing columns in cleaned training data: {missing_columns}")

data = data.dropna(subset=["label", "processed_text"]).copy()
data["processed_text"] = data["processed_text"].astype(str).str.strip()
data = data[data["processed_text"] != ""]
data["label"] = data["label"].astype(int)

bad_labels = set(data["label"].unique()) - {0, 1}
if bad_labels:
    raise ValueError(f"Unexpected labels found: {bad_labels}")

print("Cleaned data shape:", data.shape)
print(data["label"].value_counts())

###Create Train/Validation datasets

In [ ]:
X = data["processed_text"]
y = data["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Train label ratio:")
print(y_train.value_counts(normalize=True))
print("Validation label ratio:")
print(y_val.value_counts(normalize=True))

###Metrics Helpers

In [ ]:
METRIC_COLUMNS = [
    "experiment_number",
    "logged_at",
    "experiment_id",
    "model_family",
    "model_name",
    "vectorizer",
    "classifier",
    "train_accuracy",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
]

classic_results_df = pd.DataFrame(columns=METRIC_COLUMNS)

def evaluate_classifier(model_family, model_name, pipeline, X_train, y_train, X_val, y_val):
    y_train_pred = pipeline.predict(X_train)
    y_val_pred = pipeline.predict(X_val)

    result = {
        "experiment_id": datetime.now().strftime("%Y%m%d_%H%M%S_%f"),
        "model_family": model_family,
        "model_name": model_name,
        "vectorizer": pipeline.named_steps["vectorizer"].__class__.__name__,
        "classifier": pipeline.named_steps["classifier"].__class__.__name__,
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "validation_accuracy": accuracy_score(y_val, y_val_pred),
        "validation_precision": precision_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0,
        ),
        "validation_recall": recall_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0,
        ),
        "validation_f1": f1_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    return result, y_val_pred

def append_metrics(result, csv_path=classic_metrics_path):
    result = result.copy()
    result["logged_at"] = datetime.now().isoformat(timespec="seconds")

    new_row = pd.DataFrame([result])

    if csv_path.exists():
        old_rows = pd.read_csv(csv_path)
        if "experiment_number" in old_rows.columns:
            old_rows = old_rows.drop(columns=["experiment_number"])
        updated_rows = pd.concat([old_rows, new_row], ignore_index=True, sort=False)
    else:
        updated_rows = new_row

    updated_rows.insert(0, "experiment_number", range(1, len(updated_rows) + 1))

    updated_rows.to_csv(csv_path, index=False)
    return updated_rows

Every model test should call append_metrics(...) immediately after evaluation. This function updates the in-memory table returned by the function and writes the same table to:

metrics/classic_ml_experiments.csv


#Models

##Logistic Regression

###Test 1: Bag of Words

In [ ]:
logreg_bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 1),
        max_df=1.0,
        min_df=1,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

logreg_bow_pipeline.fit(X_train, y_train)

logreg_bow_result, logreg_bow_preds = evaluate_classifier(
    model_family="LogisticRegression",
    model_name="LogReg_BOW_baseline",
    pipeline=logreg_bow_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(logreg_bow_result)
classic_results_df.tail()

###Test 2: TF-IDF

In [ ]:
logreg_tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 1),
        max_df=1.0,
        min_df=1,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

logreg_tfidf_pipeline.fit(X_train, y_train)

logreg_tfidf_result, logreg_tfidf_preds = evaluate_classifier(
    model_family="LogisticRegression",
    model_name="LogReg_TFIDF_baseline",
    pipeline=logreg_tfidf_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(logreg_tfidf_result)
classic_results_df.tail()

###Test 3: Tuning Loop

In [ ]:
logreg_max_df_list = [0.7, 0.8, 0.9]
logreg_min_df_list = [1, 2, 5]
logreg_ngram_list = [(1, 1), (1, 2), (1, 3)]
logreg_C_list = [0.5, 1.0, 2.0]

logreg_tuning_results = []
best_logreg_pipeline = None
best_logreg_result = None
best_logreg_f1 = -1

for max_df, min_df, ngram_range, C in product(
    logreg_max_df_list,
    logreg_min_df_list,
    logreg_ngram_list,
    logreg_C_list,
):
    model_name = (
        f"LogReg_TFIDF_maxdf_{max_df}_mindf_{min_df}_"
        f"ngram_{ngram_range}_C_{C}"
    )

    print(f"Training {model_name}")

    pipeline = Pipeline([
        ("vectorizer", TfidfVectorizer(
            lowercase=False,
            token_pattern=r"(?u)\b\w\w+\b",
            ngram_range=ngram_range,
            max_df=max_df,
            min_df=min_df,
            norm="l2",
            use_idf=True,
            smooth_idf=True,
        )),
        ("classifier", LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=42,
        )),
    ])

    pipeline.fit(X_train, y_train)

    result, y_val_pred = evaluate_classifier(
        model_family="LogisticRegression",
        model_name=model_name,
        pipeline=pipeline,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
    )

    result["max_df"] = max_df
    result["min_df"] = min_df
    result["ngram_range"] = str(ngram_range)
    result["C"] = C

    logreg_tuning_results.append(result)
    classic_results_df = append_metrics(result)

    if result["validation_f1"] > best_logreg_f1:
        best_logreg_f1 = result["validation_f1"]
        best_logreg_result = result
        best_logreg_pipeline = pipeline

logreg_tuning_results_df = pd.DataFrame(logreg_tuning_results)

print("Best Logistic Regression result:")
print(best_logreg_result)
logreg_tuning_results_df.sort_values("validation_f1", ascending=False).head()

##Support Vector Machine (SVC)

###Test 4: BOW + LinearSVC

In [ ]:
svc_bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 1),
        max_df=1.0,
        min_df=1,
    )),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=10000,
    )),
])

svc_bow_pipeline.fit(X_train, y_train)

svc_bow_result, svc_bow_preds = evaluate_classifier(
    model_family="SVC",
    model_name="SVC_BOW_baseline",
    pipeline=svc_bow_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(svc_bow_result)
classic_results_df.tail()

###Test 5: TF-IDF + LinearSVC

In [ ]:
svc_tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 1),
        max_df=1.0,
        min_df=1,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=10000,
    )),
])

svc_tfidf_pipeline.fit(X_train, y_train)

svc_tfidf_result, svc_tfidf_preds = evaluate_classifier(
    model_family="SVC",
    model_name="SVC_TFIDF_baseline",
    pipeline=svc_tfidf_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(svc_tfidf_result)
classic_results_df.tail()

###Test 6: SVC Tuned Model

In [ ]:
svc_tuned_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 2),
        max_df=0.7,
        min_df=1,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", LinearSVC(
        C=1.0,
        random_state=42,
        max_iter=10000,
    )),
])

svc_tuned_pipeline.fit(X_train, y_train)

svc_tuned_result, svc_tuned_preds = evaluate_classifier(
    model_family="SVC",
    model_name="SVC_TFIDF_tuned_maxdf_0.7_mindf_1_ngram_1_2",
    pipeline=svc_tuned_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(svc_tuned_result)


#Keep the best SVC pipeline in memory. It will be saved only if it wins the overall classic-ML comparison.
best_svc_pipeline = svc_tuned_pipeline
classic_results_df.tail()

##Random Forest

###Test 7: RandomForest Baseline

In [ ]:
#compile the hyperparameters of the model
rf_baseline_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w\w+\b",
        ngram_range=(1, 2),
        max_df=0.7,
        min_df=1,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features="sqrt",
        criterion="gini",
        random_state=42,
        n_jobs=-1,
        class_weight=None,
    )),
])

rf_baseline_pipeline.fit(X_train, y_train)

rf_baseline_result, rf_baseline_preds = evaluate_classifier(
    model_family="RandomForest",
    model_name="RandomForest_TFIDF_baseline",
    pipeline=rf_baseline_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

classic_results_df = append_metrics(rf_baseline_result)
classic_results_df.tail()

###Test 8: Random Forest Tuning Loop

In [ ]:
n_estimators_list = [100, 200, 300]
max_depth_list = [None, 15, 30]
max_features_list = ["sqrt", "log2"]
criterion_list = ["gini", "entropy"]

rf_tuning_results = []
best_rf_pipeline = None
best_rf_result = None
best_rf_f1 = -1

for n_estimators, max_depth, max_features, criterion in product(
    n_estimators_list,
    max_depth_list,
    max_features_list,
    criterion_list,
):
    model_name = (
        f"RandomForest_TFIDF_estimators_{n_estimators}_"
        f"depth_{max_depth}_features_{max_features}_criterion_{criterion}"
    )

    print(f"Training {model_name}")

    pipeline = Pipeline([
        ("vectorizer", TfidfVectorizer(
            lowercase=False,
            token_pattern=r"(?u)\b\w\w+\b",
            ngram_range=(1, 2),
            max_df=0.7,
            min_df=1,
            norm="l2",
            use_idf=True,
            smooth_idf=True,
        )),
        ("classifier", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_features=max_features,
            criterion=criterion,
            random_state=42,
            n_jobs=-1,
            class_weight=None,
        )),
    ])

    pipeline.fit(X_train, y_train)

    result, y_val_pred = evaluate_classifier(
        model_family="RandomForest",
        model_name=model_name,
        pipeline=pipeline,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
    )

    result["n_estimators"] = n_estimators
    result["max_depth"] = str(max_depth)
    result["max_features"] = max_features
    result["criterion"] = criterion
    result["tfidf_max_df"] = 0.7
    result["tfidf_min_df"] = 1
    result["tfidf_ngram_range"] = "(1, 2)"

    rf_tuning_results.append(result)
    classic_results_df = append_metrics(result)

    if result["validation_f1"] > best_rf_f1:
        best_rf_f1 = result["validation_f1"]
        best_rf_result = result
        best_rf_pipeline = pipeline

rf_tuning_results_df = pd.DataFrame(rf_tuning_results)

print("Best Random Forest result:")
print(best_rf_result)
rf_tuning_results_df.sort_values("validation_f1", ascending=False).head()

In [ ]:
if best_rf_pipeline is None:
    best_rf_pipeline = rf_baseline_pipeline
    best_rf_result = rf_baseline_result

#Results

## Compare Results And Select The Best Model



In [ ]:
if classic_metrics_path.exists():
    all_results = pd.read_csv(classic_metrics_path)
else:
    all_results = classic_results_df.copy()

comparison = all_results.sort_values(
    by="validation_f1",
    ascending=False,
).reset_index(drop=True)

comparison.head(10)

In [ ]:
best_row = comparison.iloc[0]
best_family = best_row["model_family"]
best_model_name = best_row["model_name"]

best_pipeline_by_family = {
    "LogisticRegression": best_logreg_pipeline,
    "SVC": best_svc_pipeline,
    "RandomForest": best_rf_pipeline,
}

best_pipeline = best_pipeline_by_family[best_family]

print(f"Best model: {best_family} | {best_model_name}")

# Optional: uncomment only if you want to save the best classic ML model.
# best_model_path = project_path / "models" / "classic_ml" / "best_classic_ml_model.joblib"
# best_model_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(best_pipeline, best_model_path)

##Summarize the results

In [ ]:
top_models = comparison.head(12).copy()
top_models["plot_label"] = (
    top_models["model_family"].astype(str)
    + " | "
    + top_models["model_name"].astype(str)
)

metric_evolution_df = all_results.copy().reset_index(drop=True)

if "experiment_number" not in metric_evolution_df.columns:
    metric_evolution_df["experiment_number"] = metric_evolution_df.index + 1

In [ ]:
plt.figure(figsize=(12, 7))

plt.barh(
    top_models["plot_label"][::-1],
    top_models["validation_f1"][::-1],
)

plt.title("Top Models By Validation F1")
plt.xlabel("Validation F1")
plt.xlim(0.9, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

x_positions = range(len(top_models))

plt.plot(
    x_positions,
    top_models["train_accuracy"],
    marker="o",
    label="Train accuracy",
)

plt.plot(
    x_positions,
    top_models["validation_accuracy"],
    marker="o",
    label="Validation accuracy",
)

plt.title("Train vs Validation Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0.9, 1.0)
plt.xticks(
    ticks=list(x_positions),
    labels=top_models["model_family"],
    rotation=45,
    ha="right",
)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

for model_family, family_df in metric_evolution_df.groupby("model_family"):
    plt.plot(
        family_df["experiment_number"],
        family_df["validation_f1"],
        marker="o",
        label=model_family,
    )

plt.title("Validation F1 Evolution")
plt.xlabel("Experiment Number")
plt.ylabel("Validation F1")
plt.ylim(0.6, 1.0)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

for metric_name in [
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
]:
    plt.plot(
        metric_evolution_df["experiment_number"],
        metric_evolution_df[metric_name],
        marker="o",
        label=metric_name,
    )

plt.title("Validation Metrics Evolution")
plt.xlabel("Experiment Number")
plt.ylabel("Metric Value")
plt.ylim(0.6, 1.0)
plt.legend()

plt.tight_layout()
plt.show()

##Diagnostics

In [ ]:
validation_diagnostics_df = data.loc[X_val.index].copy()
validation_diagnostics_df["actual_label_name"] = validation_diagnostics_df["label"].map({
    0: "FAKE",
    1: "REAL",
})

model_pipelines = {
    "logistic_regression": best_logreg_pipeline,
    "svc": best_svc_pipeline,
    "random_forest": best_rf_pipeline,
}

for model_name, pipeline in model_pipelines.items():
    predictions = pipeline.predict(X_val)
    validation_diagnostics_df[f"{model_name}_prediction"] = predictions
    validation_diagnostics_df[f"{model_name}_prediction_name"] = pd.Series(
        predictions,
        index=X_val.index,
    ).map({
        0: "FAKE",
        1: "REAL",
    })
    validation_diagnostics_df[f"{model_name}_correct"] = (
        validation_diagnostics_df["label"]
        == validation_diagnostics_df[f"{model_name}_prediction"]
    )
    validation_diagnostics_df[f"{model_name}_error_type"] = "correct"
    validation_diagnostics_df.loc[
        (validation_diagnostics_df["label"] == 0)
        & (validation_diagnostics_df[f"{model_name}_prediction"] == 1),
        f"{model_name}_error_type",
    ] = "false_positive"
    validation_diagnostics_df.loc[
        (validation_diagnostics_df["label"] == 1)
        & (validation_diagnostics_df[f"{model_name}_prediction"] == 0),
        f"{model_name}_error_type",
    ] = "false_negative"

validation_diagnostics_df.to_csv(validation_diagnostics_path, index=False)
validation_diagnostics_df.head()